In [ ]:
import os
import sqlite3
import time
import re
from collections import Counter, defaultdict
import numpy as np
import matplotlib.pyplot as plt
from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import RDF
import csv
import urllib.parse

# ---------- CONFIG ----------
BASE_NAME = "YOUR_TOPIC" # name of the topic e.g. "babylon"
BASE_DIR = f"./{BASE_NAME}GPTKB" # name of the directory
seed_entity = "YOUR_SEED_ENTITY" # optional, for stats page

db_files = [
    #f"./{BASE_NAME}GPTKB.db" # could be multiple files if you have multiple runs
]

MAX_RETRIES = 100
RETRY_SLEEP = 2

output_base = os.path.join(BASE_DIR, f"{BASE_NAME}GPTKB")
os.makedirs(BASE_DIR, exist_ok=True)

# ---------- STEP 1: Load triples ----------
def get_triples_db(db_file):
    query = "SELECT subject, predicate, object FROM triple;"
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with sqlite3.connect(f'file:{db_file}?mode=ro', uri=True, timeout=30) as conn:
                cur = conn.cursor()
                cur.execute(query)
                return set(cur.fetchall())
        except sqlite3.DatabaseError as e:
            print(f"[{db_file}] attempt {attempt} failed: {e}")
            time.sleep(RETRY_SLEEP)
    return set()

triple_sets = []
for db_file in db_files:
    if os.path.exists(db_file):
        triple_sets.append(get_triples_db(db_file))
    else:
        print(f"Missing DB: {db_file}")
        triple_sets.append(set())

num_runs = len(triple_sets)

# ---------- STEP 2: Load node types ----------
def get_node_types(db_file):
    types = {}
    query = "SELECT name, type FROM node;"
    try:
        with sqlite3.connect(f'file:{db_file}?mode=ro', uri=True) as conn:
            cur = conn.cursor()
            cur.execute(query)
            for name, ntype in cur.fetchall():
                if name in types:
                    if types[name] != "instance" and ntype == "instance":
                        types[name] = "instance"
                else:
                    types[name] = ntype
    except:
        pass
    return types

node_types_all = {}
for db_file in db_files:
    if os.path.exists(db_file):
        local = get_node_types(db_file)
        for n, t in local.items():
            if n in node_types_all:
                if node_types_all[n] != "instance" and t == "instance":
                    node_types_all[n] = "instance"
            else:
                node_types_all[n] = t

# ---------- STEP 3: Count triple frequency ----------
triple_counts = Counter()
for s in triple_sets:
    for triple in s:
        triple_counts[triple] += 1

# ---------- STEP 4: Elbow detection ----------
threshold_results = {}

if num_runs >= 3:
    for X in range(1, num_runs + 1):
        threshold_results[X] = sum(1 for c in triple_counts.values() if c >= X)

    X_vals = np.array(list(threshold_results.keys()))
    Y_vals = np.array(list(threshold_results.values()))

    line_vec = np.array([X_vals[-1] - X_vals[0], Y_vals[-1] - Y_vals[0]])
    norm = np.linalg.norm(line_vec)

    if norm == 0:
        auto_X = 1
    else:
        line_vec = line_vec / norm
        distances = []
        for i in range(len(X_vals)):
            point_vec = np.array([X_vals[i] - X_vals[0], Y_vals[i] - Y_vals[0]])
            proj = np.dot(point_vec, line_vec) * line_vec
            distances.append(np.linalg.norm(point_vec - proj))
        auto_X = X_vals[np.argmax(distances)]

else:
    auto_X = 1
    print("Elbow skipped (less than 3 runs)")

final_triples = [t for t, c in triple_counts.items() if c >= auto_X]
print(f"Final triples: {len(final_triples)}")

# ---------- STEP 5: Restrict nodes ----------
used_nodes = {s for s, _, _ in final_triples} | {o for _, _, o in final_triples}

final_node_types = {
    n: node_types_all.get(n, "literal") for n in used_nodes
}

# ---------- STEP 6: Save SQLite ----------
output_db = output_base + ".db"
if os.path.exists(output_db):
    os.remove(output_db)

with sqlite3.connect(output_db) as conn:
    cur = conn.cursor()
    cur.execute("CREATE TABLE triple (subject TEXT, predicate TEXT, object TEXT)")
    cur.executemany("INSERT INTO triple VALUES (?, ?, ?)", final_triples)

    cur.execute("CREATE TABLE node (name TEXT, type TEXT)")
    cur.executemany("INSERT INTO node VALUES (?, ?)", final_node_types.items())

    cur.execute("CREATE TABLE predicate (name TEXT)")
    cur.executemany(
        "INSERT INTO predicate VALUES (?)",
        [(p,) for p in sorted({p for _, p, _ in final_triples})]
    )
    conn.commit()

# ---------- STEP 7: Save CSV ----------
output_csv = output_base + ".csv"
with open(output_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["subject", "predicate", "object"])
    writer.writerows(final_triples)

# ---------- STEP 8: Save TTL with rdf:type ----------
output_ttl = output_base + ".ttl"
g = Graph()
EX = Namespace("http://minigptkb/")

def safe_name(name):
    cleaned = name.strip().replace(" ", "_")
    return urllib.parse.quote(cleaned, safe="_")

for s, p, o in final_triples:
    subj = URIRef(EX[safe_name(s)])
    pred = URIRef(EX[safe_name(p)])

    if final_node_types.get(s) == "instance":
        g.add((subj, RDF.type, EX.Instance))

    if final_node_types.get(o) == "instance":
        obj = URIRef(EX[safe_name(o)])
    else:
        obj = Literal(o)

    g.add((subj, pred, obj))

g.serialize(destination=output_ttl, format="turtle")

# ---------- STEP 9: Basic statistics ----------

# Most frequent classes (by number of instances of that class)
# Find triples that indicate instance typing
#TYPING_PREDICATES = {'rdf:type', 'instanceof', 'type', 'is_a', 'isa', 'a'}
TYPING_PREDICATES = {'instanceof'}
class_instance_counts = Counter()
for s, p, o in final_triples:
    if p.lower() in TYPING_PREDICATES:
        class_instance_counts[o] += 1
top_classes = class_instance_counts.most_common(10)

# Most frequent predicates
pred_counter = Counter(p for _, p, _ in final_triples)
top_predicates = pred_counter.most_common(10)

csv_file = output_csv
output_dir = os.path.join(BASE_DIR, f"{BASE_NAME}GPTKB/html")
os.makedirs(output_dir, exist_ok=True)

def safe_filename(name):
    return re.sub(r'[^\w\-\. ]', '_', name)

triples = []
with open(csv_file, newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader, None)
    for row in reader:
        if len(row) == 3:
            triples.append(tuple(map(str.strip, row)))

subjects = set(s for s, _, _ in triples)
objects = set(o for _, _, o in triples)
entities = subjects.union(objects)
# Only create pages for instance entities (not literals or undefined)
page_entities = {e for e in entities if final_node_types.get(e) == "instance"}

# Compute additional stats for index page
total_entities = len(page_entities)
total_pages = len(page_entities)
total_triples = len(final_triples)

triples_by_subject = defaultdict(list)
triples_by_object = defaultdict(list)

for s, p, o in triples:
    triples_by_subject[s].append((p, o))
    triples_by_object[o].append((s, p))

def wrap_html(title, body):
    return f"""<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>{title}</title>
<style>
body {{ font-family: Arial; max-width:900px; margin:auto; }}
table {{ border-collapse:collapse; width:100%; }}
th,td {{ border:1px solid #ccc; padding:8px; }}
th {{ background:#0077cc; color:white; }}
a {{ color:#0077cc; }}
</style>
</head>
<body>
<a href="index.html">🏠 Home</a>
{body}
</body>
</html>"""

def entity_page(entity):
    outgoing = triples_by_subject.get(entity, [])
    incoming = triples_by_object.get(entity, [])

    out_rows = ""
    for p, o in outgoing:
        o_link = f'<a href="{safe_filename(o)}.html">{o}</a>' if o in page_entities else o
        out_rows += f"<tr><td>{entity}</td><td>{p}</td><td>{o_link}</td></tr>"

    in_rows = ""
    for s, p in incoming:
        s_link = f'<a href="{safe_filename(s)}.html">{s}</a>'
        in_rows += f"<tr><td>{s_link}</td><td>{p}</td><td>{entity}</td></tr>"

    body = f"<h1>{entity}</h1>"

    if outgoing:
        body += f"<h2>Outgoing</h2><table><tr><th>S</th><th>P</th><th>O</th></tr>{out_rows}</table>"

    if incoming:
        body += f"<h2>Incoming</h2><table><tr><th>S</th><th>P</th><th>O</th></tr>{in_rows}</table>"

    return wrap_html(entity, body)

for entity in page_entities:
    fname_base = safe_filename(entity)
    fname = f"{fname_base}.html"
    # Handle filename collisions (e.g. curly vs straight quotes)
    counter = 1
    original_fname = fname
    while os.path.exists(os.path.join(output_dir, fname)):
        fname = f"{fname_base}_{counter}.html"
        counter += 1
    if fname != original_fname:
        print(f"  Note: renamed '{entity}' -> {fname} (collision)")
    with open(os.path.join(output_dir, fname), "w", encoding="utf-8") as f:
        f.write(entity_page(entity))

# Build stats section
    # Count unique classes (objects where predicate=instanceOf)
    unique_classes = set(o for s, p, o in final_triples if p.lower() in TYPING_PREDICATES)

    # Count unique classes (objects where predicate=instanceOf)
    unique_classes = set(o for s, p, o in final_triples if p.lower() in TYPING_PREDICATES)

    stats_section = f"""
<h2>KB Statistics</h2>
<ul>
  <li><strong>Entities:</strong> {total_entities}</li>
  <li><strong>Predicates:</strong> {len(set(p for _, p, _ in final_triples))}</li>
  <li><strong>Literals:</strong> {len([e for e in used_nodes if final_node_types.get(e) == "literal"])}</li>
  <li><strong>Triples:</strong> {total_triples}</li>
  <li><strong>Unique Classes:</strong> {len(unique_classes)}</li>"""
if BASE_NAME:
    stats_section += f'\n  <li><strong>Seed entity:</strong> {seed_entity}</li>'
stats_section += '\n</ul>'

if top_classes:
    stats_section += '<h2>Top 10 Classes</h2><ul>'
    for cls, count in top_classes:
        stats_section += f'<li>{cls} ({count} occurrences)</li>'
    stats_section += '</ul>'

if top_predicates:
    stats_section += '<h2>Top 10 Predicates</h2><ul>'
    for pred, count in top_predicates:
        stats_section += f'<li>{pred} ({count} occurrences)</li>'
    stats_section += '</ul>'

# Build filename mapping for index links (handle collisions)
entity_to_filename = {}
for entity in page_entities:
    fname_base = safe_filename(entity)
    fname = f"{fname_base}.html"
    counter = 1
    original_fname = fname
    while fname in entity_to_filename.values():
        fname = f"{fname_base}_{counter}.html"
        counter += 1
    entity_to_filename[entity] = fname

for entity, fname in entity_to_filename.items():
    with open(os.path.join(output_dir, fname), "w", encoding="utf-8") as f:
        f.write(entity_page(entity))

index_links = "".join(
    f'<li><a href="{entity_to_filename[e]}">{e}</a></li>'
    for e in sorted(page_entities)
)

index_html = wrap_html(
    f"{BASE_NAME}GPTKB Index",
    f"<h1>{BASE_NAME}GPTKB</h1>{stats_section}<h2>Entity Index</h2><ul>{index_links}</ul>"
)

with open(os.path.join(output_dir, "index.html"), "w", encoding="utf-8") as f:
    f.write(index_html)

print(f"Generated {len(page_entities)} pages + index.html")